In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [2]:
import torch, os, argparse, accelerate, warnings
import sys
sys.path.append("/home/gaoya/Code_Video/DiffSynth-Studio-main/")
from diffsynth.core import UnifiedDataset
from diffsynth.core.data.operators import LoadVideo, LoadAudio, ImageCropAndResize, ToAbsolutePath
from diffsynth.pipelines.wan_video import WanVideoPipeline, ModelConfig
from diffsynth.diffusion import *
os.environ["TOKENIZERS_PARALLELISM"] = "false"


class WanTrainingModule(DiffusionTrainingModule):
    def __init__(
        self,
        model_paths=None, model_id_with_origin_paths=None,
        tokenizer_path=None, audio_processor_path=None,
        trainable_models=None,
        lora_base_model=None, lora_target_modules="", lora_rank=32, lora_checkpoint=None,
        preset_lora_path=None, preset_lora_model=None,
        use_gradient_checkpointing=True,
        use_gradient_checkpointing_offload=False,
        extra_inputs=None,
        fp8_models=None,
        offload_models=None,
        device="cpu",
        task="sft",
        max_timestep_boundary=1.0,
        min_timestep_boundary=0.0,
    ):
        super().__init__()
        # Warning
        if not use_gradient_checkpointing:
            warnings.warn("Gradient checkpointing is detected as disabled. To prevent out-of-memory errors, the training framework will forcibly enable gradient checkpointing.")
            use_gradient_checkpointing = True
        
        # Load models
        model_configs = self.parse_model_configs(model_paths, model_id_with_origin_paths, fp8_models=fp8_models, offload_models=offload_models, device=device)
        tokenizer_config = ModelConfig(model_id="Wan-AI/Wan2.1-T2V-1.3B", origin_file_pattern="google/umt5-xxl/") if tokenizer_path is None else ModelConfig(tokenizer_path)
        audio_processor_config = self.parse_path_or_model_id(audio_processor_path)
        self.pipe = WanVideoPipeline.from_pretrained(torch_dtype=torch.bfloat16, device=device, model_configs=model_configs, tokenizer_config=tokenizer_config, audio_processor_config=audio_processor_config)
        self.pipe = self.split_pipeline_units(task, self.pipe, trainable_models, lora_base_model)
        
        # Training mode
        self.switch_pipe_to_training_mode(
            self.pipe, trainable_models,
            lora_base_model, lora_target_modules, lora_rank, lora_checkpoint,
            preset_lora_path, preset_lora_model,
            task=task,
        )
        
        # Store other configs
        self.use_gradient_checkpointing = use_gradient_checkpointing
        self.use_gradient_checkpointing_offload = use_gradient_checkpointing_offload
        self.extra_inputs = extra_inputs.split(",") if extra_inputs is not None else []
        self.fp8_models = fp8_models
        self.task = task
        self.task_to_loss = {
            "sft:data_process": lambda pipe, *args: args,
            "direct_distill:data_process": lambda pipe, *args: args,
            "sft": lambda pipe, inputs_shared, inputs_posi, inputs_nega: FlowMatchSFTLoss(pipe, **inputs_shared, **inputs_posi),
            "sft:train": lambda pipe, inputs_shared, inputs_posi, inputs_nega: FlowMatchSFTLoss(pipe, **inputs_shared, **inputs_posi),
            "direct_distill": lambda pipe, inputs_shared, inputs_posi, inputs_nega: DirectDistillLoss(pipe, **inputs_shared, **inputs_posi),
            "direct_distill:train": lambda pipe, inputs_shared, inputs_posi, inputs_nega: DirectDistillLoss(pipe, **inputs_shared, **inputs_posi),
        }
        self.max_timestep_boundary = max_timestep_boundary
        self.min_timestep_boundary = min_timestep_boundary
        
    def parse_extra_inputs(self, data, extra_inputs, inputs_shared):
        for extra_input in extra_inputs:
            if extra_input == "input_image":
                inputs_shared["input_image"] = data["video"][0]
            elif extra_input == "end_image":
                inputs_shared["end_image"] = data["video"][-1]
            elif extra_input == "reference_image" or extra_input == "vace_reference_image":
                inputs_shared[extra_input] = data[extra_input][0]
            else:
                inputs_shared[extra_input] = data[extra_input]
        if inputs_shared.get("framewise_decoding", False):
            # WanToDance global model
            inputs_shared["num_frames"] = 4 * (len(data["video"]) - 1) + 1
        return inputs_shared
    
    def get_pipeline_inputs(self, data):
        inputs_posi = {"prompt": data["prompt"]}
        inputs_nega = {}
        inputs_shared = {
            # Assume you are using this pipeline for inference,
            # please fill in the input parameters.
            "input_video": data["video"],
            "height": data["video"][0].size[1],
            "width": data["video"][0].size[0],
            "num_frames": len(data["video"]),
            # Please do not modify the following parameters
            # unless you clearly know what this will cause.
            "cfg_scale": 1,
            "tiled": False,
            "rand_device": self.pipe.device,
            "use_gradient_checkpointing": self.use_gradient_checkpointing,
            "use_gradient_checkpointing_offload": self.use_gradient_checkpointing_offload,
            "cfg_merge": False,
            "vace_scale": 1,
            "max_timestep_boundary": self.max_timestep_boundary,
            "min_timestep_boundary": self.min_timestep_boundary,
        }
        inputs_shared = self.parse_extra_inputs(data, self.extra_inputs, inputs_shared)
        return inputs_shared, inputs_posi, inputs_nega
    
    def forward(self, data, inputs=None):
        if inputs is None: inputs = self.get_pipeline_inputs(data)
        inputs = self.transfer_data_to_device(inputs, self.pipe.device, self.pipe.torch_dtype)
        for unit in self.pipe.units:
            inputs = self.pipe.unit_runner(unit, self.pipe, *inputs)
        loss = self.task_to_loss[self.task](self.pipe, *inputs)
        return loss





/data/gaoya/miniconda3/envs/wan/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
I0000 00:00:1776581267.962560 2554331 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776581270.679603 2554331 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776581275.086087 2554331 port.cc:153] oneDNN custom operations are on. You may see slightly different n

In [3]:
import os
import json
import argparse

def wan_parser(argv=None):
    parser = argparse.ArgumentParser(
        description="Simple example of a training script.",
        allow_abbrev=False,
    )
    parser = add_general_config(parser)
    parser = add_video_size_config(parser)
    parser.add_argument("--tokenizer_path", type=str, default=None, help="Path to tokenizer.")
    parser.add_argument("--audio_processor_path", type=str, default=None, help="Path to the audio processor. If provided, the processor will be used for Wan2.2-S2V model.")
    parser.add_argument("--max_timestep_boundary", type=float, default=1.0, help="Max timestep boundary.")
    parser.add_argument("--min_timestep_boundary", type=float, default=0.0, help="Min timestep boundary.")
    parser.add_argument("--initialize_model_on_cpu", default=False, action="store_true", help="Whether to initialize models on CPU.")
    parser.add_argument("--framewise_decoding", default=False, action="store_true", help="Enable it if this model is a WanToDance global model.")
    args, _ = parser.parse_known_args(args=argv)
    return args


WAN_ROOT = "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B"

# tokenizer 路径
tokenizer_candidates = [
    os.path.join(WAN_ROOT, "google", "umt5-xxl"),
    os.path.join(WAN_ROOT, "google"),
]
tokenizer_path = next((p for p in tokenizer_candidates if os.path.isdir(p)), None)
if tokenizer_path is None:
    raise FileNotFoundError(
        f"没找到 tokenizer 目录。请检查 {WAN_ROOT}/google 或 {WAN_ROOT}/google/umt5-xxl"
    )

# VAE 路径
VAE_PATH = os.path.join(WAN_ROOT, "Wan2.2_VAE.pth")
if not os.path.isfile(VAE_PATH):
    raise FileNotFoundError(
        f"缺少 VAE 权重：{VAE_PATH}"
    )

# 关键修复：DiT 分片必须作为“一个子列表”传入
dit_shards = [
    os.path.join(WAN_ROOT, "diffusion_pytorch_model-00001-of-00003.safetensors"),
    os.path.join(WAN_ROOT, "diffusion_pytorch_model-00002-of-00003.safetensors"),
    os.path.join(WAN_ROOT, "diffusion_pytorch_model-00003-of-00003.safetensors"),
]

for p in dit_shards:
    if not os.path.isfile(p):
        raise FileNotFoundError(f"缺少 DiT 分片：{p}")

t5_path = os.path.join(WAN_ROOT, "models_t5_umt5-xxl-enc-bf16.pth")
if not os.path.isfile(t5_path):
    raise FileNotFoundError(f"缺少 T5 权重：{t5_path}")

local_model_paths = [
    dit_shards,   # 必须是嵌套 list
    t5_path,
    VAE_PATH,
]

args = wan_parser([
    "--dataset_base_path", "/home/gaoya/Code_Video/Code_data/Code_train/assets",
    "--dataset_metadata_path", "/home/gaoya/Code_Video/Code_data/Code_train/assets/metadata.csv",
    "--height", "480",
    "--width", "832",
    "--num_frames", "49",
    "--dataset_repeat", "100",
    "--model_paths", json.dumps(local_model_paths),
    "--tokenizer_path", tokenizer_path,
    "--learning_rate", "1e-4",
    "--num_epochs", "5",
    "--remove_prefix_in_ckpt", "pipe.dit.",
    "--output_path", "/data/gaoya/AAA_test_video/Train_test/DiffSynth_wan22_ti2v5B/0323/models/train/Wan2.2-TI2V-5B_lora",
    "--lora_base_model", "dit",
    "--lora_target_modules", "q,k,v,o,ffn.0,ffn.2",
    "--lora_rank", "32",
    "--extra_inputs", "input_image",
])

print("args.model_paths =", args.model_paths)
print("parsed json =", json.loads(args.model_paths))
print(args)

args.model_paths = [["/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003.safetensors", "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003.safetensors", "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003.safetensors"], "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/models_t5_umt5-xxl-enc-bf16.pth", "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/Wan2.2_VAE.pth"]
parsed json = [['/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003.safetensors', '/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003.safetensors', '/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003.safetensors'], '/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/models_t5_umt5-xxl-enc-bf16.pth', '/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/Wan2.2_VAE.pth']
Namespace(dataset_base_path='/home/gaoya/Code_Video/Code_data/Code_train/assets', dataset_metadata_path='/home/gaoya/Code_Video/Code_data/Cod

In [4]:
accelerator = accelerate.Accelerator(
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    kwargs_handlers=[
        accelerate.DistributedDataParallelKwargs(
            find_unused_parameters=args.find_unused_parameters
        )
    ],
    log_with="wandb",
)
# accelerator = accelerate.Accelerator(
#     gradient_accumulation_steps=args.gradient_accumulation_steps,
#     kwargs_handlers=[accelerate.DistributedDataParallelKwargs(find_unused_parameters=args.find_unused_parameters)],
# )
dataset = UnifiedDataset(
    base_path=args.dataset_base_path,
    metadata_path=args.dataset_metadata_path,
    repeat=args.dataset_repeat,# 100
    data_file_keys=args.data_file_keys.split(","),
    main_data_operator=UnifiedDataset.default_video_operator(
        base_path=args.dataset_base_path,
        max_pixels=args.max_pixels,
        height=args.height,
        width=args.width,
        height_division_factor=16,
        width_division_factor=16,
        num_frames=args.num_frames,
        time_division_factor=4 if not args.framewise_decoding else 1,
        time_division_remainder=1 if not args.framewise_decoding else 0,
    ),
    special_operator_map={
        "animate_face_video": ToAbsolutePath(args.dataset_base_path) >> LoadVideo(args.num_frames, 4, 1, frame_processor=ImageCropAndResize(512, 512, None, 16, 16)),
        "input_audio": ToAbsolutePath(args.dataset_base_path) >> LoadAudio(sr=16000),
        "wantodance_music_path": ToAbsolutePath(args.dataset_base_path),
    }
)


model = WanTrainingModule(
    model_paths=args.model_paths,
    model_id_with_origin_paths=args.model_id_with_origin_paths,
    tokenizer_path=args.tokenizer_path,
    audio_processor_path=args.audio_processor_path,
    trainable_models=args.trainable_models,
    lora_base_model=args.lora_base_model,
    lora_target_modules=args.lora_target_modules,
    lora_rank=args.lora_rank,
    lora_checkpoint=args.lora_checkpoint,
    preset_lora_path=args.preset_lora_path,
    preset_lora_model=args.preset_lora_model,
    use_gradient_checkpointing=args.use_gradient_checkpointing,
    use_gradient_checkpointing_offload=args.use_gradient_checkpointing_offload,
    extra_inputs=args.extra_inputs,
    fp8_models=args.fp8_models,
    offload_models=args.offload_models,
    task=args.task,
    device="cpu" if args.initialize_model_on_cpu else accelerator.device,
    max_timestep_boundary=args.max_timestep_boundary,
    min_timestep_boundary=args.min_timestep_boundary,
)



/tmp/ipykernel_2554331/2041916036.py:32: UserWarning: Gradient checkpointing is detected as disabled. To prevent out-of-memory errors, the training framework will forcibly enable gradient checkpointing.
  warnings.warn("Gradient checkpointing is detected as disabled. To prevent out-of-memory errors, the training framework will forcibly enable gradient checkpointing.")


Loading models from: [
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00001-of-00003.safetensors",
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00002-of-00003.safetensors",
    "/data/gaoya/ckpt/Wan-AI-Wan2.2-TI2V-5B/diffusion_pytorch_model-00003-of-00003.safetensors"
]
Loaded model: {
    "model_name": "wan_video_dit",
    "model_class": "diffsynth.models.wan_video_dit.WanModel",
    "extra_kwargs": {
        "has_image_input": false,
        "patch_size": [
            1,
            2,
            2
        ],
        "in_dim": 48,
        "dim": 3072,
        "ffn_dim": 14336,
        "freq_dim": 256,
        "text_dim": 4096,
        "out_dim": 48,
        "num_heads": 24,
        "num_layers": 30,
        "eps": 1e-06,
        "seperated_timestep": true,
        "require_clip_embedding": false,
        "require_vae_embedding": false,
        "fuse_vae_embedding_in_latents": true
    }
}
Loading models from: "/data/gaoya/ckpt/Wan-AI-Wan2

In [5]:
import random

import wandb
import os
os.environ['WANDB_API_KEY'] = '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'
# 要加apikey

accelerator.init_trackers(
    project_name="wan-train",
    config={
        "learning_rate": args.learning_rate,
        "num_epochs": args.num_epochs,
        "height": args.height,
        "width": args.width,
        "num_frames": args.num_frames,
        "lora_rank": args.lora_rank,
        "lora_target_modules": args.lora_target_modules,
        "dataset_repeat": args.dataset_repeat,
        "output_path": args.output_path,
    },
    init_kwargs={
        "wandb": {
            # 改成你当前账号真正有写权限的 entity
            "entity": "875222004-gy",
            # run 名建议用 basename，不要用 dirname
            "name": os.path.basename(args.output_path.rstrip("/")),
        }
    },
)
model_logger = ModelLogger(
    args.output_path,
    remove_prefix_in_ckpt=args.remove_prefix_in_ckpt,
)
launcher_map = {
    "sft:data_process": launch_data_process_task,
    "direct_distill:data_process": launch_data_process_task,
    "sft": launch_training_task,
    "sft:train": launch_training_task,
    "direct_distill": launch_training_task,
    "direct_distill:train": launch_training_task,
}




wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 875222004 (875222004-gy) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:

# launcher_map[args.task](accelerator, dataset, model, model_logger, args=args)


In [6]:
from diffsynth.diffusion.runner import initialize_deepspeed_gradient_checkpointing
from tqdm import tqdm
learning_rate = args.learning_rate
weight_decay = args.weight_decay
num_workers = args.dataset_num_workers
save_steps = args.save_steps
num_epochs = args.num_epochs
optimizer = torch.optim.AdamW(model.trainable_modules(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.ConstantLR(optimizer)
dataloader = torch.utils.data.DataLoader(dataset, shuffle=True, collate_fn=lambda x: x[0], num_workers=num_workers)
model.to(device=accelerator.device)
model, optimizer, dataloader, scheduler = accelerator.prepare(model, optimizer, dataloader, scheduler)
initialize_deepspeed_gradient_checkpointing(accelerator)


In [7]:
global_step = 0

for epoch_id in range(num_epochs):
    model.train()
    for data in tqdm(dataloader):
        with accelerator.accumulate(model):
            optimizer.zero_grad()

            if dataset.load_from_cache:
                loss = model({}, inputs=data)
            else:
                loss = model(data)

            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()

            # 只在真正同步更新时记一次，避免梯度累积时重复记
            if accelerator.sync_gradients:
                global_step += 1

                accelerator.log(
                    {
                        "train/loss": loss.detach().float().item(),
                        "train/lr": scheduler.get_last_lr()[0],
                        "train/epoch": epoch_id,
                    },
                    step=global_step,
                )

            model_logger.on_step_end(accelerator, model, save_steps, loss=loss)

    # 每个 epoch 额外记一次
    accelerator.log(
        {
            "train/epoch_end": epoch_id,
        },
        step=global_step,
    )

    if save_steps is None:
        model_logger.on_epoch_end(accelerator, model, epoch_id)

model_logger.on_training_end(accelerator, model, save_steps)
accelerator.end_training()

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [08:08<00:00,  4.88s/it]


train/epoch,▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆████████
train/epoch_end,▁▃▅▆█
train/loss,▆█▂▂▃▁▂▆▂▁▁▂▂▇▂▂▂▁▂▂▆▂▃▂▂▂▁▆▁▁▁▃▂▁▁▁▂▁▁▁
train/lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,4
train/epoch_end,4
train/loss,0.08031
train/lr,0.0001
